# uMedSum baseline on the simulated consultations: full SOAP (S, O and A/P)

Runs the original uMedSum pipeline on the whole note and scores the whole note. **Baseline only, no comparison.**

| | |
|---|---|
| Data | `agentic-pipeline-output/soap_notes`: 207 folders, of which **201 are usable**. D2N030, D2N051, D2N086, D2N101, D2N108 and D2N110 have an empty `consultation.md` |
| Split | fresh reshuffle, seed 42 → 40 dev / 161 test. Only used as a label; scores are on all cases |
| D2N002 | uMedSum's 1-shot example, so it is run but left out of scoring → **200 scored cases** |
| Stage 1 | Element-Aware + 1-shot ICL full SOAP note (LLaMA-3-8B Q8) |
| Stage 2 | RTB-TS confabulation removal on **Subjective, Objective and Assessment/Plan** |
| Stage 3 | MMR/KeyBERT missing-information detection + GPT-2 insertion over the **whole note** |
| Reference | the agentic `soap_note.md`: Subjective, Objective, Assessment + Plan |
| Scores | ROUGE / BERTScore (full note, Subjective, A/P), grounding of every S/O/A/P sentence in the consultation, fact recall against facts from the full reference note, Stage 3 task metrics |

Every case is generated from scratch. Output: `Baseline_Implementation/umedsum_final_soap_notes/output/`.

## 1. Mount Drive and set paths

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, glob
from collections import Counter
ROOT = "/content/drive/MyDrive/tokenizers"
MODEL = "llama3:8b-instruct-q8_0"
EXCLUDE = ["D2N002"]                         # 1-shot ICL demonstration: run, not scored
EMPTY_CONSULTATION = {"D2N030", "D2N051", "D2N086", "D2N101", "D2N108", "D2N110"}

SOAP_NOTES = f"{ROOT}/agentic-pipeline-output/soap_notes"
UM_CODE  = f"{ROOT}/Baseline_Implementation/umedsum_final_soap_notes"
UM_OUT   = f"{UM_CODE}/output"
DATA_DIR = f"{UM_OUT}/data"
DATASET  = f"{DATA_DIR}/prepared_dataset.json"
PREP_CODE = f"{ROOT}/verify_and_ask_pipeline/stage1_pipeline/stage1_code"   # data preparation script only
EVAL_CODE = f"{ROOT}/verify_and_ask_pipeline/Stage23_pipeline/code"         # evaluator only

for p in [SOAP_NOTES, f"{PREP_CODE}/prepare_dataset.py", f"{EVAL_CODE}/evaluate.py",
          f"{ROOT}/Baseline_Implementation/umedsum_pipeline_baseline/stage_2/confabulation_remover.py",
          f"{ROOT}/Baseline_Implementation/aci_dataset_rq1_baseline/icl_examples.json"]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)
cases = sorted(d for d in os.listdir(SOAP_NOTES) if os.path.isdir(f"{SOAP_NOTES}/{d}"))
print("case folders:", len(cases))
old = [p for p in glob.glob(f"{UM_OUT}/cases/*.json") if json.load(open(p)).get("scope") != "soap"]
assert not old, f"{len(old)} case files from the old S/O run are still in {UM_OUT}/cases - delete the output folder first"
for d in (UM_CODE, UM_OUT, DATA_DIR, f"{UM_OUT}/cases", f"{UM_OUT}/evaluation"):
    os.makedirs(d, exist_ok=True)

Mounted at /content/drive
OK      /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes
OK      /content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/stage1_pipeline/stage1_code/prepare_dataset.py
OK      /content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/Stage23_pipeline/code/evaluate.py
OK      /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_pipeline_baseline/stage_2/confabulation_remover.py
OK      /content/drive/MyDrive/tokenizers/Baseline_Implementation/aci_dataset_rq1_baseline/icl_examples.json
case folders: 207


## 2. Write the scripts into Drive
`run_umedsum_soap.py` (now with `--scope soap`) and `evaluate_full_soap.py` (the project evaluator extended to S, O and A/P; `evaluate.py` itself is not modified).

In [2]:
%%writefile /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes/run_umedsum_soap.py
"""
uMedSum (Nagar et al., ACL 2025) on the 110 simulated consultations, matched to the RQ1 setup.

This is baseline B3 from the proposal ("Aegle intake -> note -> uMedSum filter"), run end to end:

  Stage 1  Element-Aware + 1-shot ICL full-SOAP generation (LLaMA-3-8B)   <- same prompts as the baseline
           -> split into S/O and A/P. A/P is kept aside and never changed.
  Stage 2  RTB-TS confabulation removal (DeBERTa-v3-large NLI, Te=0.9, Tc=0.8, Ta=0.5)   on S/O only
  Stage 3  MMR key sentences + KeyBERT coverage (cov_min=0.4, top_m=2) + GPT-2 perplexity insertion   on S/O only
  Final    Stage 3 S/O + the untouched Stage 1 A/P

Input is the same prepared dataset Verify-and-Ask uses (built from agentic-pipeline-output/soap_notes
by verify_and_ask_pipeline/stage1_pipeline/stage1_code/prepare_dataset.py, seed 42; pass it with --dataset). Only the consultation is given to the models; the agentic soap_note.md S/O is
carried in each record solely for evaluation.

Output records use the Verify-and-Ask record format (stage1_so / stage2_so / stage3_so, stage1_ap,
ap_unchanged, stage3_added, ...), so verify_and_ask_pipeline/Stage23_pipeline/code/evaluate.py scores
both systems with exactly the same code. Systems A/B/C in that evaluation = uMedSum Stage 1/2/3.

The Stage 1/2/3 algorithms are imported unchanged from Baseline_Implementation/umedsum_pipeline_baseline.
The only adaptation is section handling (S/O vs A/P), plus an optional fix to the baseline's header
detection (see FixedHeaderRemover), on by default.

  python run_umedsum_soap.py --limit 2            # smoke test
  python run_umedsum_soap.py                      # all 110 cases (resumes per case)
  python run_umedsum_soap.py --split test         # only the 88 test cases
"""
import argparse
import json
import re
import sys
import time
import traceback
from pathlib import Path

HERE = Path(__file__).resolve().parent
REPO = HERE.parent.parent  # .../tokenizers
BASELINE = REPO / "Baseline_Implementation" / "umedsum_pipeline_baseline"
for sub in ("stage_1", "stage_2", "stage_3"):
    sys.path.insert(0, str(BASELINE / sub))

DEFAULT_DATASET = REPO / "verify_and_ask_pipeline" / "stage1_pipeline" / "stage1_output" / "prepared_dataset.json"
DEFAULT_ICL = REPO / "Baseline_Implementation" / "aci_dataset_rq1_baseline" / "icl_examples.json"
DEFAULT_OUT = HERE / "output"


# ---------------------------------------------------------------------------------------------
# Section handling
# ---------------------------------------------------------------------------------------------
_MAIN_HEADING = re.compile(
    r"^(?P<name>subjective|objective|assessment\s*(?:and|&|/)\s*plan|assessment|plan|a\s*/\s*p)\b"
    r"(?:\s*\([^)]*\))?\s*(?P<colon>:)?\s*(?P<rest>.*)$",
    re.IGNORECASE,
)
# Used only if the model wrote no SUBJECTIVE / OBJECTIVE / ASSESSMENT headings at all.
_FALLBACK_SUBHEADINGS = [
    ("subjective", r"chief complaint|history of present illness|hpi|review of systems|past medical history|"
                   r"medications?|allergies|social history|family history|cc"),
    ("objective", r"physical exam(?:ination)?|exam(?:ination)?|vitals?|vital signs|diagnostic|results?|labs?|"
                  r"imaging|investigations?"),
    ("ap", r"diagnos[ie]s|impression|clinical reasoning|treatment|follow[- ]?up|recommendations?"),
]


def _strip_markup(line: str) -> str:
    s = line.strip()
    s = re.sub(r"^#{1,6}\s*", "", s)
    s = re.sub(r"^(?:[-*+•]|\d+[.)]|[IVX]+[.)])\s+", "", s)
    s = s.replace("**", "").replace("__", "").strip()
    return s


def _main_heading(line: str):
    """Return (section, rest_of_line) if the line opens a SOAP section, else None."""
    s = _strip_markup(line)
    m = _MAIN_HEADING.match(s)
    if not m:
        return None
    rest = m.group("rest").strip()
    # "Subjective findings include ..." is prose, not a heading: need a colon or nothing after the name.
    if rest and not m.group("colon"):
        return None
    name = m.group("name").lower().replace(" ", "")
    section = "subjective" if name.startswith("subj") else "objective" if name.startswith("obj") else "ap"
    return section, rest.strip("* ")


def _fallback_heading(line: str):
    s = _strip_markup(line)
    head, _, rest = s.partition(":")
    head = head.strip()
    if not head or len(head.split()) > 6:
        return None
    if not (s.endswith(":") or rest):  # needs "Heading:" or "Heading: text"
        return None
    for section, pat in _FALLBACK_SUBHEADINGS:
        if re.fullmatch(rf"(?:{pat})(?:\s*\(.*\))?(?:\s*(?:&|and)\s*.*)?", head, re.IGNORECASE):
            return section, rest.strip()
    return None


_PREAMBLE = re.compile(r"^(here is|here's|below is|this is the|soap note|sure[,!.]|certainly)", re.IGNORECASE)


def split_umedsum_note(note: str) -> dict:
    """Split a Stage 1 uMedSum note into S, O and A/P.

    The baseline prompt asks for 'SUBJECTIVE:', 'OBJECTIVE:', 'ASSESSMENT AND PLAN:'. Text before the
    first heading (chat preambles) is dropped. If no main headings are found, the baseline's own
    sub-headings (Chief Complaint, Physical Examination, ...) are used instead.
    """
    out = {"subjective": [], "objective": [], "ap": [], "preamble": []}
    lines = note.splitlines()
    found = {"subjective": False, "objective": False, "ap": False}
    for detector in (_main_heading, _fallback_heading):
        out = {"subjective": [], "objective": [], "ap": [], "preamble": []}
        found = {"subjective": False, "objective": False, "ap": False}
        current = "preamble"
        for line in lines:
            h = detector(line)
            if h:
                current, rest = h
                found[current] = True
                if rest:
                    out[current].append(rest)
                continue
            out[current].append(line.rstrip())
        if found["subjective"] or found["objective"]:
            mode = "main_headings" if detector is _main_heading else "subheadings"
            break
    else:
        # No recognisable headings: keep the whole note as S/O so nothing is silently lost (flagged).
        mode = "unparsed"
        out = {"subjective": lines, "objective": [], "ap": [], "preamble": []}
    clean = lambda xs: "\n".join(xs).strip()
    return {
        "subjective": clean(out["subjective"]),
        "objective": clean(out["objective"]),
        "assessment_and_plan": clean(out["ap"]),
        "dropped_preamble": clean(out["preamble"]),
        "parse_mode": mode,
        "parse_ok": mode != "unparsed" and found["ap"],
    }


# ---------------------------------------------------------------------------------------------
# Stage 2 helper: the baseline's unit splitter, with the header bug fixed
# ---------------------------------------------------------------------------------------------
def make_remover_class(fix_header_bug: bool):
    from confabulation_remover import ConfabulationRemover

    class SectionRemover(ConfabulationRemover):
        """Runs RTB-TS on one section at a time.

        The baseline's clean_summary_text() treats ANY line that contains 'EXAM', 'PLAN', 'OBJECTIVE',
        'ASSESSMENT', 'SUBJECTIVE', 'CHIEF COMPLAINT' or 'HISTORY OF PRESENT ILLNESS' as a heading and
        drops it before NLI scoring (e.g. 'Physical examination revealed ...', 'He explained ...').
        Its bullet regex also strips leading numbers ('50-year-old' -> '-year-old'). With
        fix_header_bug=True only short heading-like lines are skipped and only real bullets are
        stripped; scoring and thresholds are unchanged. Set False to reproduce the baseline exactly.
        """
        forced_section = "SUBJECTIVE"

        def clean_summary_text(self, text):
            if not fix_header_bug:
                units = super().clean_summary_text(text)
                for u in units:
                    u["section"] = self.forced_section
                return units
            units = []
            for line in text.splitlines():
                s = line.strip()
                if not s:
                    continue
                hdr = _strip_markup(s).rstrip(":").strip()
                if (":" not in hdr and len(hdr.split()) <= 6 and not re.search(r"[.!?]$", hdr) and (
                        s.endswith(":") or _main_heading(hdr) or _fallback_heading(hdr + ":"))):
                    continue  # a bare heading such as 'Chief Complaint:' or 'Physical Examination & Vitals'
                if re.match(r"^(here is|here's|below is|this is the|soap note:?)", s, re.IGNORECASE):
                    continue
                if re.match(r"^(please note|note:|disclaimer:)", s, re.IGNORECASE):
                    continue
                body = re.sub(r"^(?:[-*+•]|\d+[.)])\s+", "", s).replace("**", "").replace("__", "").strip()
                if len(body.split()) < 3:
                    continue
                for seg in self.segmenter.segment(body):
                    seg = seg.strip()
                    if len(seg.split()) >= 3:
                        units.append({"unit": seg, "section": self.forced_section})
            return units

    return SectionRemover


# ---------------------------------------------------------------------------------------------
# Stage 3 helper: Algorithm 2 (GPT-2 perplexity insertion) over S/O with section tracking
# ---------------------------------------------------------------------------------------------
def perplexity_insert(merger, units, missing):
    """units: list of (section, sentence). Each missing sentence goes where total perplexity is lowest
    (baseline Algorithm 2). Its section is that of the preceding sentence (or the next one at index 0)."""
    units = list(units)
    log = []
    for sent in missing:
        if not units:
            units.append(("subjective", sent))
            log.append({"inserted_sentence": sent, "location_index": 0, "section": "subjective", "best_ppl": None})
            continue
        best_loc, best_ppl = 0, float("inf")
        texts = [u[1] for u in units]
        for loc in range(len(texts) + 1):
            ppl = merger.calculate_perplexity(" ".join(texts[:loc] + [sent] + texts[loc:]))
            if ppl < best_ppl:
                best_loc, best_ppl = loc, ppl
        section = units[best_loc - 1][0] if best_loc > 0 else units[0][0]
        units.insert(best_loc, (section, sent))
        log.append({"inserted_sentence": sent, "location_index": best_loc, "section": section,
                    "best_ppl": round(best_ppl, 2)})
    return units, log


def units_to_so(units, sections=("subjective", "objective")):
    return {sec: "\n".join(t for s, t in units if s == sec) for sec in sections}


# ---------------------------------------------------------------------------------------------
# Per-case pipeline
# ---------------------------------------------------------------------------------------------
def run_case(rec, ctx, args):
    from stage1_generator import run_stage1_element_aware

    t0 = time.time()
    conv = rec["conversation"]
    out = {"case_id": rec["case_id"], "split": rec["split"], "is_icl_demo": rec["case_id"] in ctx["icl_ids"]}

    full = args.scope == "soap"
    sections = ("subjective", "objective", "assessment_and_plan") if full else ("subjective", "objective")

    # ---- Stage 1: unchanged baseline generator (Element-Aware 2-step + 1-shot ICL) ----------------
    s1 = None
    cache_p = Path(args.stage1_cache) / f"{rec['case_id']}.json" if args.stage1_cache else None
    if cache_p and cache_p.exists():
        prev = json.loads(cache_p.read_text(encoding="utf-8"))
        if prev.get("status") == "success" and prev.get("stage1_raw_note"):
            s1 = {"stage1_summary": prev["stage1_raw_note"], "stage1_elements": prev.get("stage1_elements")}
            out["stage1_reused_from"] = str(cache_p)
    if s1 is None:
        s1 = run_stage1_element_aware(ctx["client"], {"src": conv}, icl_prefix=ctx["icl_prefix"])
    parsed = split_umedsum_note(s1["stage1_summary"])
    stage1_so = {sec: parsed[sec] for sec in sections}
    stage1_ap = {"assessment_and_plan": parsed["assessment_and_plan"]}
    out.update({
        "stage1_elements": s1["stage1_elements"],
        "stage1_raw_note": s1["stage1_summary"],
        "stage1_parse": {k: parsed[k] for k in ("parse_mode", "parse_ok", "dropped_preamble")},
        "stage1_so": stage1_so,
        "stage1_ap": stage1_ap,
    })

    # ---- Stage 2: RTB-TS section by section (S, O; plus A/P in --scope soap) ------------------------
    remover = ctx["remover"]
    stage2_units, details, removed, n_claims, n_kept = [], {}, [], 0, 0
    for section in sections:
        text = stage1_so[section]
        if not text.strip():
            details[section] = []
            continue
        remover.forced_section = section.upper()
        res = remover.process_summary(conv, text)
        stage2_units += [(section, u) for u in res["faithful_units"]]
        details[section] = res["details"]
        for d in res["details"]:
            n_claims += 1
            kept = d["classification"] == "faithful" or bool(d.get("atomic_decomposition", {}).get("retained"))
            n_kept += int(kept)
            if d["classification"] == "confabulated":
                removed.append({"section": section, "claim": d["unit"], "scores": d["scores"], "reason": d["reasoning"]})
            for a in d.get("atomic_decomposition", {}).get("discarded", []):
                removed.append({"section": section, "claim": a["atom"], "entailment": a["entailment"],
                                "reason": a["reasoning"], "from_unit": d["unit"]})
    stage2_so = units_to_so(stage2_units, sections)
    counts = {}
    for sec in details.values():
        for d in sec:
            counts[d["classification"]] = counts.get(d["classification"], 0) + 1
    out.update({
        "stage2_so": stage2_so,
        "stage2_audit": details,
        "stage2_stats": {"decision_counts": counts, "num_claims": n_claims, "num_kept": n_kept,
                         "claim_retention_rate": round(n_kept / n_claims, 4) if n_claims else None},
    })

    # ---- Stage 3: missing-information addition on S/O --------------------------------------------
    extractor, detector, merger = ctx["extractor"], ctx["detector"], ctx["merger"]
    key_sents = extractor.extract_document_key_sentences(conv, top_m=args.top_m)
    so_flat = " ".join(t for _, t in stage2_units)
    phrases = extractor.extract_summary_key_phrases(so_flat, top_n=10)
    detection = detector.detect_missing_sentences(key_sents, phrases)
    missing = detection["missing_sentences"]
    if args.no_gpt2:
        final_units = stage2_units + [("subjective", m) for m in missing]
        ins_log = [{"inserted_sentence": m, "location_index": None, "section": "subjective", "best_ppl": None}
                   for m in missing]
    else:
        final_units, ins_log = perplexity_insert(merger, stage2_units, missing)
    stage3_so = units_to_so(final_units, sections)
    cov = {c["sentence"]: c["score"] for c in detection["coverage_scores"]}
    out.update({
        "stage3_so": stage3_so,
        "stage3_key_sentences": key_sents,
        "stage3_summary_key_phrases": phrases,
        "stage3_coverage_scores": detection["coverage_scores"],
        # Same shape as Verify-and-Ask so evaluate.py computes gap-detection / addition metrics.
        "stage3_candidate_log": [{"candidate": m, "statement": m, "coverage": cov.get(m), "status": "added",
                                  "reason": "coverage <= cov_min (uMedSum adds the dialogue sentence verbatim)",
                                  "insertion": {"section": l["section"], "line_index": l["location_index"],
                                                "perplexity": l["best_ppl"]}}
                                 for m, l in zip(missing, ins_log)],
        "stage3_added": [{"statement": m, "evidence": m, "section": l["section"], "candidate": m}
                         for m, l in zip(missing, ins_log)],
        "stage3_stats": {"num_key_sentences": len(key_sents), "num_flagged": len(missing), "num_added": len(missing)},
        "gaps": {"gaps": [], "dropped": [], "num_generated": 0},  # uMedSum has no Ask stage
        "unsupported_claims": {"removed_stage2": removed},
    })

    # ---- Final note ------------------------------------------------------------------------------
    if full:   # A/P went through Stages 2-3 like S and O (original uMedSum: the whole note is processed)
        final_ap = {"assessment_and_plan": stage3_so["assessment_and_plan"]}
        out.update({"final_ap": final_ap, "scope": "soap",
                    "ap_changed": final_ap["assessment_and_plan"] != stage1_ap["assessment_and_plan"]})
        ref = rec.get("reference_soap") or {}
        reference = {"subjective": ref.get("subjective", ""), "objective": ref.get("objective", ""),
                     "assessment_and_plan": "\n".join(x for x in (ref.get("assessment", ""), ref.get("plan", "")) if x).strip()}
    else:      # Stage 3 S/O + untouched Stage 1 A/P
        final_ap = dict(stage1_ap)
        out.update({"final_ap": final_ap, "scope": "so",
                    "ap_unchanged": final_ap == stage1_ap and final_ap["assessment_and_plan"] == parsed["assessment_and_plan"]})
        reference = rec["reference_so"]
    out.update({
        "thresholds": {"Te": args.te, "Tc": args.tc, "Ta": args.ta, "cov_min": args.cov_min, "top_m": args.top_m,
                       "fix_header_bug": not args.reproduce_header_bug},
        "models": {"llm": args.model, "nli": args.nli_model, "embedder": "all-MiniLM-L6-v2", "ppl": args.ppl_model},
        "status": "success",
        "seconds": round(time.time() - t0, 1),
        "conversation": conv,
        "reference_so": reference,  # evaluation only; never passed to a model (full SOAP in --scope soap)
    })
    return out


def write_final_note(path, r):
    so, ap = r["stage3_so"], r["final_ap"]["assessment_and_plan"]
    ap_title = "Assessment and Plan" if r.get("scope") == "soap" else "Assessment and Plan (Stage 1, unchanged)"
    path.write_text(
        f"# {r['case_id']} - uMedSum final note\n\n## Subjective\n{so['subjective'] or '(empty)'}\n\n"
        f"## Objective\n{so['objective'] or '(empty)'}\n\n## {ap_title}\n{ap or '(empty)'}\n",
        encoding="utf-8")


def main():
    ap = argparse.ArgumentParser(description="uMedSum on the agentic SOAP-note consultations (RQ1-matched)")
    ap.add_argument("--dataset", default=str(DEFAULT_DATASET))
    ap.add_argument("--icl-json", default=str(DEFAULT_ICL))
    ap.add_argument("--n-shots", type=int, default=1)
    ap.add_argument("--out-dir", default=str(DEFAULT_OUT))
    ap.add_argument("--split", choices=["dev", "test", "all"], default="all")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--model", default="llama3:8b")
    ap.add_argument("--ollama-url", default="http://localhost:11434")
    ap.add_argument("--nli-model", default="MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli")
    ap.add_argument("--ppl-model", default="gpt2")
    ap.add_argument("--te", type=float, default=0.9)
    ap.add_argument("--tc", type=float, default=0.8)
    ap.add_argument("--ta", type=float, default=0.5)
    ap.add_argument("--cov-min", type=float, default=0.4)
    ap.add_argument("--top-m", type=int, default=2)
    ap.add_argument("--no-gpt2", action="store_true", help="append missing sentences instead of GPT-2 placement")
    ap.add_argument("--reproduce-header-bug", action="store_true",
                    help="use the baseline's original line filter in Stage 2 (drops lines containing EXAM/PLAN/...)")
    ap.add_argument("--exclude-from-scoring", nargs="*", default=[],
                    help="case ids that are run but kept out of scoring (their split becomes 'excluded_icl_demo'); "
                         "use for an ICL demonstration that falls in the test split, e.g. D2N002")
    ap.add_argument("--scope", choices=["so", "soap"], default="so",
                    help="so: Stages 2-3 on S/O, A/P passed through (RQ1-matched). "
                         "soap: Stages 2-3 on S, O and A/P, all scored (original uMedSum)")
    ap.add_argument("--stage1-cache", default=None,
                    help="folder of earlier per-case outputs (cases/) whose Stage 1 note is reused instead of regenerated")
    ap.add_argument("--retry-errors", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()

    out_dir = Path(args.out_dir)
    cases_dir, notes_dir = out_dir / "cases", out_dir / "final_notes"
    cases_dir.mkdir(parents=True, exist_ok=True)
    notes_dir.mkdir(parents=True, exist_ok=True)

    data = json.loads(Path(args.dataset).read_text(encoding="utf-8"))
    records = data["records"]
    n_dev = sum(r["split"] == "dev" for r in records)
    n_test = sum(r["split"] == "test" for r in records)
    print(f"Dataset: {len(records)} cases ({n_dev} dev / {n_test} test), seed {data.get('split_seed')}")

    # ---- leakage guard: the ICL demonstration must not be a test case ----------------------------
    icl_data = json.loads(Path(args.icl_json).read_text(encoding="utf-8"))["data"][: args.n_shots]
    icl_ids = {d.get("encounter_id") or d.get("file", "").split("-")[0] for d in icl_data}
    excluded = set(args.exclude_from_scoring)
    split_of = {r["case_id"]: ("excluded_icl_demo" if r["case_id"] in excluded else r["split"]) for r in records}
    leaked = {c for c in icl_ids if split_of.get(c) == "test"}
    if leaked:
        sys.exit(f"[LEAKAGE] ICL demonstration(s) {sorted(leaked)} are test cases. "
                 f"Pass --exclude-from-scoring {' '.join(sorted(leaked))} to run them but keep them out of scoring.")
    print(f"ICL demonstration(s): {sorted(icl_ids)} - none scored in the test split. "
          f"Excluded from scoring: {sorted(excluded) or 'none'}. "
          f"Demo cases by split: { {c: split_of.get(c, 'not in dataset') for c in sorted(icl_ids)} }")

    if args.split != "all":
        records = [r for r in records if r["split"] == args.split]
    if args.limit:
        records = records[: args.limit]

    # ---- models -----------------------------------------------------------------------------------
    from ollama_client import OllamaClient
    from prompts import format_icl_demonstrations
    from nli_scorer import DebertaNLIScorer
    from atomic_decomposer import AtomicDecomposer
    from key_extractor import KeyExtractor
    from missing_detector import MissingInformationDetector
    from perplexity_merger import PerplexityMerger

    client = OllamaClient(base_url=args.ollama_url, model=args.model)
    if not client.check_connection():
        sys.exit(f"Ollama is not reachable at {args.ollama_url}. Start `ollama serve` and pull {args.model}.")
    if not any(args.model in m for m in client.list_models()):
        sys.exit(f"Model {args.model} is not pulled. Run: ollama pull {args.model}")
    Remover = make_remover_class(fix_header_bug=not args.reproduce_header_bug)
    extractor = KeyExtractor()
    ctx = {
        "client": client,
        "icl_prefix": format_icl_demonstrations(icl_data, n_shots=args.n_shots),
        "icl_ids": icl_ids,
        "remover": Remover(DebertaNLIScorer(model_name=args.nli_model),
                           AtomicDecomposer(ollama_url=args.ollama_url, model=args.model),
                           te=args.te, tc=args.tc, ta=args.ta),
        "extractor": extractor,
        "detector": MissingInformationDetector(embedder=extractor.embedder, cov_min=args.cov_min),
        "merger": None if args.no_gpt2 else PerplexityMerger(model_name=args.ppl_model),
    }

    # ---- run, one checkpoint file per case ---------------------------------------------------------
    t_start = time.time()
    for i, rec in enumerate(records, 1):
        cpath = cases_dir / f"{rec['case_id']}.json"
        if cpath.exists() and not args.overwrite:
            prev = json.loads(cpath.read_text(encoding="utf-8"))
            if prev.get("status") == "success" and prev.get("scope", "so") != args.scope:
                sys.exit(f"{cpath} was produced with --scope {prev.get('scope', 'so')}; use a different --out-dir "
                         f"for --scope {args.scope} (pass the old cases/ folder as --stage1-cache to reuse Stage 1).")
            if prev.get("status") == "success" or not args.retry_errors:
                print(f"[{i}/{len(records)}] {rec['case_id']}: cached ({prev.get('status')})")
                continue
        print(f"[{i}/{len(records)}] {rec['case_id']} ({rec['split']}) ...", end="", flush=True)
        try:
            r = run_case(rec, ctx, args)
            write_final_note(notes_dir / f"{rec['case_id']}.md", r)
            st = r["stage2_stats"]
            print(f" done {r['seconds']}s | parse={r['stage1_parse']['parse_mode']} | "
                  f"S2 kept {st['num_kept']}/{st['num_claims']} | S3 added {r['stage3_stats']['num_added']}")
        except Exception as exc:
            r = {"case_id": rec["case_id"], "split": rec["split"], "status": "error",
                 "error": f"{type(exc).__name__}: {exc}", "traceback": traceback.format_exc()}
            print(f" FAILED: {r['error']}")
        cpath.write_text(json.dumps(r, ensure_ascii=False, indent=2), encoding="utf-8")

    # ---- combine every case checkpoint into one file (the evaluator's input) ----------------------
    all_recs = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(cases_dir.glob("*.json"))]
    # Split labels always come from the dataset in use (cached cases may carry a label from an older split).
    all_recs = [r for r in all_recs if r["case_id"] in split_of]
    for r in all_recs:
        r["split"] = split_of[r["case_id"]]
    ok = [r for r in all_recs if r.get("status") == "success"]
    combined = {"system": "uMedSum (Element-Aware+1-shot ICL -> RTB-TS -> MMR/GPT-2), " + (
                    "full SOAP: S, O and A/P processed" if args.scope == "soap" else "S/O only, A/P passthrough"),
                "scope": args.scope,
                "num_cases": len(all_recs), "num_success": len(ok), "num_errors": len(all_recs) - len(ok),
                "icl_demo_ids": sorted(icl_ids), "excluded_from_scoring": sorted(excluded),
                "split_counts": {k: sum(r["split"] == k for r in all_recs) for k in sorted(set(split_of.values()))},
                "records": all_recs}
    (out_dir / "umedsum_output.json").write_text(json.dumps(combined, ensure_ascii=False, indent=2), encoding="utf-8")

    parse_modes = {}
    for r in ok:
        m = r["stage1_parse"]["parse_mode"] + ("" if r["stage1_parse"]["parse_ok"] else " (no A/P heading)")
        parse_modes[m] = parse_modes.get(m, 0) + 1
    print(f"\nFinished in {round(time.time() - t_start)}s. {len(ok)}/{len(all_recs)} cases succeeded.")
    print("Stage 1 section parsing:", parse_modes)
    if ok and all("ap_unchanged" in r for r in ok):
        print("A/P unchanged rate:", round(sum(r["ap_unchanged"] for r in ok) / len(ok), 4))
    elif ok:
        print("A/P changed by Stages 2-3 in", sum(bool(r.get("ap_changed")) for r in ok), "of", len(ok), "cases (scope soap)")
    print("Saved", out_dir / "umedsum_output.json")


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes/run_umedsum_soap.py


In [3]:
%%writefile /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes/evaluate_full_soap.py
"""
Full-SOAP evaluation of uMedSum: the project's evaluator (verify_and_ask_pipeline/Stage23_pipeline/code/evaluate.py),
extended from S/O to Subjective, Objective AND Assessment/Plan.

evaluate.py itself is not modified. This script loads its source, applies a few targeted changes, and runs it:
  * the note text used everywhere (ROUGE/BERTScore, fact extraction, fact recall) includes Assessment and Plan
  * grounding scores every sentence of S, O and A/P against the consultation
  * reference metrics are reported for the full note ("soap_"), Subjective ("subj_") and A/P ("ap_")
  * reference facts are extracted from the full reference SOAP note (S, O, A and P)

Records must come from run_umedsum_soap.py --scope soap (stage*_so and reference_so carry "assessment_and_plan").
Paths and settings come from the evaluator's config, so set RQ1_OUTPUT_DIR to the uMedSum output folder:
results go to <RQ1_OUTPUT_DIR>/evaluation/results_<split>.json/.md.

  RQ1_OUTPUT_DIR=<out> python evaluate_full_soap.py --eval-code <.../Stage23_pipeline/code> --stage23-json <file> --split all
"""
import argparse
import sys
import types
from pathlib import Path

SO_TO_TEXT_FULL = '''
def so_to_text(so: dict) -> str:
    """Full SOAP dict -> one text block (Subjective, Objective, Assessment and Plan)."""
    parts = []
    for key, title in (("subjective", "Subjective"), ("objective", "Objective"),
                       ("assessment_and_plan", "Assessment and Plan")):
        parts.append(f"{title}:\\n{(so.get(key) or '').strip()}")
    return "\\n\\n".join(parts).strip()
'''

PATCHES = [
    # full-note text everywhere so_to_text is used
    ("from common import read_json, so_to_text, write_json\n",
     "from common import read_json, write_json\n" + SO_TO_TEXT_FULL),
    # grounding: every section, A/P included
    ('for section in ("subjective", "objective"):', 'for section in ("subjective", "objective", "assessment_and_plan"):'),
    # reference metrics: full note, Subjective, A/P
    ('views = {"so_": (lambda x: so_to_text(x), "S/O"),',
     'views = {"soap_": (lambda x: so_to_text(x), "Full SOAP"),\n'
     '             "ap_": (lambda x: (x.get("assessment_and_plan") or "").strip(), "Assessment and Plan"),'),
    # facts from the whole reference note
    ('"Reference Subjective/Objective note:\\n{note}\\n\\n"',
     '"Reference SOAP note (Subjective, Objective, Assessment and Plan):\\n{note}\\n\\n"'),
    # markdown summary
    ('"Similarity to the ground-truth S/O (higher is better)"', '"Similarity to the ground-truth SOAP note (higher is better)"'),
    ('"so_rouge1_f", "so_rouge2_f", "so_rougeLsum_f", "so_bertscore_f1"]',
     '"soap_rouge1_f", "soap_rouge2_f", "soap_rougeLsum_f", "soap_bertscore_f1",\n'
     '          "ap_rouge1_f", "ap_rouge2_f", "ap_rougeLsum_f", "ap_bertscore_f1"]'),
    ('"Fact coverage vs the ground-truth S/O (higher is better)"', '"Fact coverage vs the ground-truth SOAP note (higher is better)"'),
    ('"A/P is not part of any score; S/O only."', '"Full SOAP: Subjective, Objective and Assessment/Plan are all scored."'),
    ('"Reference S/O used only for reference metrics and fact extraction."',
     '"Reference SOAP note (S, O, A/P) used only for reference metrics and fact extraction."'),
]


def main():
    ap = argparse.ArgumentParser(add_help=False)
    ap.add_argument("--eval-code", required=True, help="path to verify_and_ask_pipeline/Stage23_pipeline/code")
    known, rest = ap.parse_known_args()
    code_dir = Path(known.eval_code).resolve()
    sys.path.insert(0, str(code_dir))

    src = (code_dir / "evaluate.py").read_text(encoding="utf-8")
    for old, new in PATCHES:
        if src.count(old) != 1:
            sys.exit(f"evaluate.py has changed; could not apply patch for: {old[:60]!r}")
        src = src.replace(old, new)

    mod = types.ModuleType("evaluate_full_soap_patched")
    mod.__file__ = str(code_dir / "evaluate.py")
    sys.argv = [sys.argv[0]] + rest
    exec(compile(src, mod.__file__, "exec"), mod.__dict__)
    mod.main()


if __name__ == "__main__":
    main()

Writing /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes/evaluate_full_soap.py


## 3. Install dependencies, start Ollama, pull the model
Re-run after any disconnect.

In [4]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip install -q torch transformers accelerate sentencepiece protobuf sentence-transformers keybert pysbd numpy requests rouge-score bert-score pandas

import subprocess, time, torch
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {MODEL}
os.environ.update({"OLLAMA_MODEL": MODEL, "LLM_BASE_URL": "", "NLI_BASE_URL": "", "EMBED_BASE_URL": "",
                   "PYTHONIOENCODING": "utf-8"})
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to GPU")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.

## 4. Build the dataset (fresh reshuffle, seed 42)
Stores the consultation (model input) and all four reference sections from `soap_note.md` (evaluation only).

In [5]:
os.environ.update({"DATASET_ROOT": SOAP_NOTES, "STAGE1_OUTPUT_DIR": DATA_DIR, "SPLIT_SEED": "42", "DEV_FRACTION": "0.2"})
%cd {PREP_CODE}
!python prepare_dataset.py
d = json.load(open(DATASET))
split = {r["case_id"]: r["split"] for r in d["records"]}
skipped = sorted(set(cases) - set(split))
print("cases:", len(split), dict(Counter(split.values())), "| D2N002:", split.get("D2N002"))
print("skipped (empty consultation):", skipped)
assert set(skipped) == EMPTY_CONSULTATION, f"unexpected skipped cases: {sorted(set(skipped) ^ EMPTY_CONSULTATION)}"
missing_ap = [r["case_id"] for r in d["records"] if not (r["reference_soap"].get("assessment") or r["reference_soap"].get("plan"))]
print("reference notes without Assessment/Plan:", missing_ap or "none")
print(f"usable: {len(split)} | scored after excluding D2N002: {len(split) - len(EXCLUDE)}")

/content/drive/MyDrive/tokenizers/verify_and_ask_pipeline/stage1_pipeline/stage1_code
[WARN] Empty conversation: /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes/D2N030/consultation.md
[WARN] Empty conversation: /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes/D2N051/consultation.md
[WARN] Empty conversation: /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes/D2N086/consultation.md
[WARN] Empty conversation: /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes/D2N101/consultation.md
[WARN] Empty conversation: /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes/D2N108/consultation.md
[WARN] Empty conversation: /content/drive/MyDrive/tokenizers/agentic-pipeline-output/soap_notes/D2N110/consultation.md
Prepared 201 cases (40 dev / 161 test); skipped 6.
Output: /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes/output/data/prepared_dataset.json
cases: 201 {'test': 161,

## 5. Run uMedSum on the full note (all 201 cases, from scratch)
Resumable: re-running continues from the last finished case. Add `--retry-errors` to redo failed cases.

In [6]:
%cd {UM_CODE}
!python run_umedsum_soap.py --scope soap --dataset {DATASET} --out-dir {UM_OUT} --model {MODEL} --exclude-from-scoring {" ".join(EXCLUDE)}

/content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes
Dataset: 201 cases (40 dev / 161 test), seed 42
ICL demonstration(s): ['D2N002'] - none scored in the test split. Excluded from scoring: ['D2N002']. Demo cases by split: {'D2N002': 'excluded_icl_demo'}
modules.json: 100% 349/349 [00:00<00:00, 1.37MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 558kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 26.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 300kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.92MB/s]

model.safetensors: downloading bytes:  80% 72.3M/90.9M [00:01<00:00, 63.9MB/s, 6.08MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:02<00:00, 40.5MB/s, 7.89MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:02<00:00, 43.2MB/s, 8.65MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 6191.96it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.72MB/s]
vocab.txt: 100% 232k/232k [00:00<

## 6. Check the run and build the scoring file (D2N002 removed)

In [7]:
um = json.load(open(f"{UM_OUT}/umedsum_output.json"))
ok = [r for r in um["records"] if r.get("status") == "success"]
print(f"success {len(ok)}/{len(split)} | errors {sorted(r['case_id'] for r in um['records'] if r.get('status') != 'success')} "
      f"| not run {sorted(set(split) - {r['case_id'] for r in um['records']})}")
print("scope:", Counter(r.get("scope") for r in ok))
print("A/P changed by Stages 2-3:", sum(bool(r.get("ap_changed")) for r in ok), "/", len(ok))
print("Stage 1 section parsing:", dict(Counter(r["stage1_parse"]["parse_mode"] for r in ok)))

SCORING = f"{UM_OUT}/umedsum_output_scoring.json"
json.dump({**um, "records": [r for r in um["records"] if r["case_id"] not in EXCLUDE]}, open(SCORING, "w"), ensure_ascii=False, indent=2)
print("scoring file:", sum(r["case_id"] not in EXCLUDE for r in um["records"]), "cases")

success 201/201 | errors [] | not run []
scope: Counter({'soap': 201})
A/P changed by Stages 2-3: 201 / 201
Stage 1 section parsing: {'main_headings': 201}
scoring file: 200 cases


## 7. Score the full SOAP note (all scored cases)
Stage 1 / Stage 2 / Stage 3 appear as A_stage1 / B_verified / C_final. Writes `output/evaluation/results_all.json/.md`.

In [8]:
os.environ["RQ1_OUTPUT_DIR"] = UM_OUT
%cd {UM_CODE}
!python evaluate_full_soap.py --eval-code {EVAL_CODE} --stage23-json {SCORING} --split all

/content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes
Phase 1/3: ROUGE / BERTScore on 200 cases (loads roberta-large on CPU; a few minutes)
config.json: 100% 482/482 [00:00<00:00, 2.11MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 122kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 1.81MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 1.46MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 22.0MB/s]

model.safetensors: downloading bytes:   8% 107M/1.42G [00:01<00:11, 110MB/s, 4.80MB/s  ]  
model.safetensors: downloading bytes:  12% 171M/1.42G [00:01<00:06, 187MB/s, 10.3MB/s  ]
model.safetensors: downloading bytes:  28% 401M/1.42G [00:02<00:02, 412MB/s, 31.0MB/s  ]
model.safetensors: reconstructing file:  28% 402M/1.42G [00:02<00:04, 248MB/s, 25.4MB/s  ]
model.safetensors: downloading bytes:  47% 672M/1.42G [00:02<00:01, 592MB/s, 53.9MB/s  ]
model.safetensors: reconstructing file:  57% 804M/1.42G [00:02<00:01, 503MB/s, 61.2MB/s  ]
model.safetensors: d

## 8. Baseline scores

In [9]:
import pandas as pd
from IPython.display import display, Markdown
res = json.load(open(f"{UM_OUT}/evaluation/results_all.json"))
names = {"A_stage1": "Stage 1 (generation)", "B_verified": "Stage 2 (RTB-TS)", "C_final": "Stage 3 final"}
groups = [
    ("Overlap with reference: full SOAP", ["soap_rouge1_f", "soap_rouge2_f", "soap_rougeL_f", "soap_rougeLsum_f", "soap_bertscore_f1"]),
    ("Overlap with reference: Subjective", ["subj_rouge1_f", "subj_rouge2_f", "subj_rougeL_f", "subj_rougeLsum_f", "subj_bertscore_f1"]),
    ("Overlap with reference: Assessment/Plan", ["ap_rouge1_f", "ap_rouge2_f", "ap_rougeL_f", "ap_rougeLsum_f", "ap_bertscore_f1"]),
    ("Grounding in the consultation (S, O, A/P)", ["num_claims", "supported_rate", "uncertain_rate", "unsupported_claim_rate",
                                                   "contradiction_rate", "mean_entailment"]),
    ("Reference fact recall (full SOAP)", ["atomic_fact_recall", "critical_fact_recall", "recall_of_recoverable_facts"]),
]
rows = [{"group": g, "metric": k, **{names[s]: res["per_system"][s].get(k) for s in names}} for g, keys in groups for k in keys]
scores = pd.DataFrame(rows).set_index(["group", "metric"])
task_keys = ["claim_retention_rate_stage2", "valid_claim_retention_proxy", "num_genuine_omissions", "gap_detection_recall",
             "gap_detection_precision", "fact_recovery_rate", "num_added", "unsupported_addition_rate", "duplicate_addition_rate"]
task = pd.DataFrame([{"metric": k, "uMedSum": res["task_metrics"].get(k)} for k in task_keys]).set_index("metric")
scores.to_csv(f"{UM_OUT}/evaluation/umedsum_scores_full_soap.csv")
task.to_csv(f"{UM_OUT}/evaluation/umedsum_task_metrics_full_soap.csv")

display(Markdown(f"## uMedSum baseline, full SOAP: {res['num_cases']} cases (201 usable, D2N002 excluded)"))
display(scores.style.format(lambda v: f"{v:.4f}" if isinstance(v, float) else v, na_rep="-"))
display(Markdown("### Stage 3 task and integrity metrics"))
display(task)
print("saved:", f"{UM_OUT}/evaluation/umedsum_scores_full_soap.csv", "and umedsum_task_metrics_full_soap.csv")

## uMedSum baseline, full SOAP: 200 cases (201 usable, D2N002 excluded)

### Stage 3 task and integrity metrics

,uMedSum
metric,
claim_retention_rate_stage2,0.9819
valid_claim_retention_proxy,0.8198
num_genuine_omissions,693.0000
gap_detection_recall,0.0188
gap_detection_precision,0.1395
fact_recovery_rate,0.0794
num_added,86.0000
unsupported_addition_rate,0.1744
duplicate_addition_rate,0.4419


saved: /content/drive/MyDrive/tokenizers/Baseline_Implementation/umedsum_final_soap_notes/output/evaluation/umedsum_scores_full_soap.csv and umedsum_task_metrics_full_soap.csv


## Saved outputs (`Baseline_Implementation/umedsum_final_soap_notes/output/`)
| Path | Content |
|---|---|
| `data/prepared_dataset.json` | 201 cases with the seed-42 split |
| `cases/<case>.json`, `final_notes/<case>.md` | per-case outputs and final full notes |
| `umedsum_output.json` / `umedsum_output_scoring.json` | all cases / the 200 scored cases |
| `evaluation/results_all.json/.md` | evaluator output |
| `evaluation/umedsum_scores_full_soap.csv`, `umedsum_task_metrics_full_soap.csv` | the tables above |